# PCA on six greenhouse sensors

Six temperature sensors in a greenhouse log a reading every hour, and the six readings move together because two things drive them: the sun and the ventilation fan. Here you build principal component analysis from the covariance matrix in NumPy, find that two components carry nearly all of the variance in the six readings, and rebuild the readings from two numbers an hour. The lesson's four points, where every number comes out exact, check each step on the way.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It builds the lesson's four points and 500 hours of readings from the six sensors, with a seeded random generator so that every run gives the same numbers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# The lesson's four points. Every number PCA computes on them is exact, so the
# checks use them to test each step.
X_small = np.array([[0, 0], [2, 2], [3, 1], [1, 3]], dtype=float)

# Six greenhouse sensors, one reading an hour for 500 hours. Two hidden signals
# drive them: how strong the sun is (a daily cycle, zero at night) and how hard
# the ventilation fan is running. Each sensor responds to both with its own
# weights, plus its own noise.
rng = np.random.default_rng(0)
n_hours = 500
hour = np.arange(n_hours)
sun = 6 * np.clip(np.sin(2 * np.pi * (hour - 6) / 24), 0, None) + rng.normal(0, 0.5, n_hours)
fan = rng.normal(0, 2, n_hours)
loadings = np.array([[1.0, 1.2, 0.8, 1.5, 0.6, 1.1],       # degrees per unit of sun, per sensor
                     [-0.2, -1.0, 0.1, -1.5, -0.8, 0.3]])  # degrees per unit of fan, per sensor
X = 20 + np.column_stack([sun, fan]) @ loadings + rng.normal(0, 0.4, (n_hours, 6))
print("readings:", X.shape, "  first hour:", X[0])

plt.figure(figsize=(9, 3))
for j in range(6):
    plt.plot(hour[:72], X[:72, j], lw=1, label=f"sensor {j}")
plt.xlabel("hour")
plt.ylabel("degrees C")
plt.title("three days of readings")
plt.legend(ncol=6, fontsize=8)
plt.show()

## Part 1 (coding task): Center the data

PCA is about variance around the mean, so the first step is to subtract the mean of each column. Write `center(X)`, which returns the pair `(mu, Xc)`: `mu` holds the mean of each column, one number per sensor, and `Xc` is `X` with `mu` subtracted from every row. `X.mean(axis=0)` averages down the columns, and NumPy subtracts a vector from every row of a matrix on its own.

For the lesson's four points the mean is $(1.5, 1.5)$ and the centered points are $(-1.5, -1.5)$, $(0.5, 0.5)$, $(1.5, -0.5)$ and $(-0.5, 1.5)$.

In [ ]:
def center(X):
    """(mu, Xc): the column means, and X with them subtracted from every row."""
    # TODO
    raise NotImplementedError

In [ ]:
mu_small, Xc_small = center(X_small)
print("mean of the four points:", mu_small)
print("centered points:\n", Xc_small)
assert np.allclose(mu_small, [1.5, 1.5]), f"the mean of the four points is (1.5, 1.5), got {mu_small}"
assert np.allclose(Xc_small, [[-1.5, -1.5], [0.5, 0.5], [1.5, -0.5], [-0.5, 1.5]]), "subtract the mean from every row"

mu, Xc = center(X)
assert mu.shape == (6,), f"one mean per sensor, so mu should have shape (6,), got {mu.shape}"
assert Xc.shape == X.shape, "Xc should have the same shape as X"
assert np.allclose(Xc.mean(axis=0), 0), "every column of Xc should average to zero"
assert np.allclose(Xc + mu, X), "Xc + mu should give X back"
print("sensor means:", mu)
print("Part 1 passed")

## Part 2 (coding task): The covariance matrix

With the data centered, the sample covariance matrix is $\Sigma = X_c^\top X_c / (n - 1)$, where $n$ is the number of rows. Entry $(j, k)$ is the covariance of column $j$ with column $k$, and the diagonal holds each column's variance. Write `covariance(Xc)`, which takes centered data and returns that matrix. `Xc.T @ Xc` adds up the products over all rows in one call, and `len(Xc)` is $n$.

For the four points the matrix is $\begin{bmatrix} 5/3 & 1/3 \\ 1/3 & 5/3 \end{bmatrix}$. Dividing by $n$ instead of $n - 1$ gives different numbers, and the check catches it.

In [ ]:
def covariance(Xc):
    """The sample covariance matrix of centered data, divided by n - 1."""
    # TODO
    raise NotImplementedError

In [ ]:
Sigma_small = covariance(Xc_small)
print("covariance of the four points:\n", Sigma_small)
assert Sigma_small.shape == (2, 2), f"two columns give a 2 x 2 matrix, got {Sigma_small.shape}"
assert not np.allclose(Sigma_small, [[1.25, 0.25], [0.25, 1.25]]), "divide by n - 1 = 3, not by n = 4"
assert np.allclose(Sigma_small, [[5 / 3, 1 / 3], [1 / 3, 5 / 3]]), f"expected [[5/3, 1/3], [1/3, 5/3]], got {Sigma_small}"

Sigma = covariance(Xc)
assert Sigma.shape == (6, 6), f"six sensors give a 6 x 6 matrix, got {Sigma.shape}"
assert np.allclose(Sigma, Sigma.T), "a covariance matrix is symmetric"
assert np.allclose(np.diag(Sigma), Xc.var(axis=0, ddof=1)), "the diagonal should hold each sensor's variance"
assert np.allclose(Sigma, np.cov(Xc, rowvar=False)), "this should match np.cov(Xc, rowvar=False)"
print("sensor variances:", np.diag(Sigma))
print("total variance:", round(float(Sigma.trace()), 4))
print("Part 2 passed")

## Part 3 (coding task): The principal components

The direction of greatest variance is the eigenvector of $\Sigma$ with the largest eigenvalue, and that eigenvalue is the variance along it. The second component is the eigenvector with the next largest eigenvalue, and so on. Write `principal_components(Sigma)`, which returns `(vals, vecs)`: the eigenvalues in descending order and the matching unit eigenvectors as the columns of `vecs`, so that `vecs[:, 0]` is the first component.

`np.linalg.eigh` is the routine for a symmetric matrix. It returns the eigenvalues in ascending order, smallest first, with the eigenvectors as columns in the same order, so both have to be reordered. Forget that and the first component is the direction of least variance. For the four points the eigenvalues are 2 and $4/3$ and the components are $(1, 1)/\sqrt{2}$ and $(1, -1)/\sqrt{2}$. The sign of an eigenvector is arbitrary, so the checks ignore it.

In [ ]:
def principal_components(Sigma):
    """(vals, vecs): eigenvalues in descending order, unit eigenvectors as the columns of vecs."""
    # TODO
    raise NotImplementedError

In [ ]:
vals_small, vecs_small = principal_components(Sigma_small)
print("eigenvalues of the four points:", vals_small)
print("components as columns:\n", vecs_small)
assert np.allclose(vals_small, [2, 4 / 3]), f"the eigenvalues are 2 and 4/3, largest first, got {vals_small}"
r = 1 / np.sqrt(2)
assert np.allclose(np.abs(vecs_small[:, 0]), [r, r]), "the first component should be (1, 1) / sqrt(2), up to sign"
assert np.allclose(np.abs(vecs_small[:, 1]), [r, r]) and np.isclose(vecs_small[0, 1] * vecs_small[1, 1], -0.5), "the second component should be (1, -1) / sqrt(2), up to sign"

vals, vecs = principal_components(Sigma)
assert vals.shape == (6,) and vecs.shape == (6, 6), "six eigenvalues and a 6 x 6 matrix of eigenvectors"
assert np.all(np.diff(vals) <= 1e-12), "the eigenvalues should be in descending order"
assert np.allclose(vecs.T @ vecs, np.eye(6)), "the columns should be unit vectors, perpendicular to each other"
for j in range(6):
    assert np.allclose(Sigma @ vecs[:, j], vals[j] * vecs[:, j]), f"column {j} is not an eigenvector of Sigma"
assert np.isclose(vals.sum(), Sigma.trace()), "the eigenvalues should add up to the total variance"
print("eigenvalues of the sensor data:", vals)
print("Part 3 passed")

## Part 4 (coding task): How much each component explains

Each eigenvalue is the variance along its component, and together they add up to the total variance, so component $j$ explains the fraction $\lambda_j / \sum_i \lambda_i$. Write `explained(vals)`, which returns `(fraction, cumulative)`: the fraction each component explains and the running total, which `np.cumsum` computes. Then write `components_for(cumulative, target)`, the smallest number of components whose running total reaches `target`. `np.searchsorted(cumulative, target)` finds the first position where the running total is at least `target`, and positions start at 0 while counts start at 1.

The four points give fractions 0.6 and 0.4.

In [ ]:
def explained(vals):
    """(fraction, cumulative): the share of variance per component and its running total."""
    # TODO
    raise NotImplementedError


def components_for(cumulative, target):
    """The smallest k whose first k components explain at least `target` of the variance."""
    # TODO
    raise NotImplementedError

In [ ]:
fraction_small, cumulative_small = explained(vals_small)
assert np.allclose(fraction_small, [0.6, 0.4]), f"the four points give fractions 0.6 and 0.4, got {fraction_small}"
assert np.allclose(cumulative_small, [0.6, 1.0]), f"the running total should be [0.6, 1.0], got {cumulative_small}"
assert components_for(cumulative_small, 0.5) == 1, "one component already explains 0.6, so 0.5 needs one"
assert components_for(cumulative_small, 0.6) == 1, "the first component reaches 0.6 exactly, so 0.6 needs one component"
assert components_for(cumulative_small, 0.9) == 2, "0.9 needs both components"

fraction, cumulative = explained(vals)
for k in range(6):
    print(f"{k + 1} components: {fraction[k]:.4f} of the variance, {cumulative[k]:.4f} so far")
k95 = components_for(cumulative, 0.95)
assert np.isclose(cumulative[-1], 1.0), "all six components together explain everything"
assert cumulative[0] < 0.95 < cumulative[1], "the sensor data has two strong directions, and the first alone is not enough for 95%"
assert k95 == 2, f"two components reach 95% on the sensor data, got {k95}"
print("components for 95%:", k95)

plt.figure(figsize=(5, 3))
plt.plot(range(1, 7), cumulative, "o-")
plt.axhline(0.95, ls="--", color="gray")
plt.xlabel("components kept")
plt.ylabel("cumulative variance explained")
plt.ylim(0, 1.02)
plt.show()
print("Part 4 passed")

## Part 5 (coding task): Two numbers an hour

The scores are the coordinates of the centered points along the components, $Z = X_c W$, where the columns of $W$ are the components kept. Write `project(Xc, vecs, k)`, which returns the $n \times k$ matrix of scores on the first `k` components. Each score is the dot product of a centered row with a component, and one matrix multiplication does all of them.

For the four points the scores on the first component are $-3/\sqrt{2}$ and three times $1/\sqrt{2}$, up to the sign of the component. For the sensors, two scores an hour replace six readings, and the check plots them.

In [ ]:
def project(Xc, vecs, k):
    """Scores of the centered rows on the first k components, an (n, k) matrix."""
    # TODO
    raise NotImplementedError

In [ ]:
z_small = project(Xc_small, vecs_small, 1)
print("scores of the four points on the first component:", z_small.ravel())
assert z_small.shape == (4, 1), f"four points and one component give shape (4, 1), got {z_small.shape}"
assert np.allclose(np.abs(z_small.ravel()), [3 / np.sqrt(2), 1 / np.sqrt(2), 1 / np.sqrt(2), 1 / np.sqrt(2)]), "expected -3/sqrt(2) and three times 1/sqrt(2), up to sign"

Z = project(Xc, vecs, 2)
assert Z.shape == (500, 2), f"500 hours and two components give shape (500, 2), got {Z.shape}"
assert np.allclose(Z.mean(axis=0), 0), "scores of centered data average to zero"
assert np.allclose(np.cov(Z, rowvar=False), np.diag(vals[:2])), "the two score columns should be uncorrelated, with the eigenvalues as their variances"
print("correlation of the scores with the hidden signals:")
for j in range(2):
    print(f"  component {j + 1}: with sun {np.corrcoef(Z[:, j], sun)[0, 1]:+.3f}, with fan {np.corrcoef(Z[:, j], fan)[0, 1]:+.3f}")

plt.figure(figsize=(5, 4))
plt.scatter(Z[:, 0], Z[:, 1], c=sun, s=8, cmap="viridis")
plt.colorbar(label="sun")
plt.xlabel("score on component 1")
plt.ylabel("score on component 2")
plt.title("500 hours as two numbers each")
plt.show()
print("Part 5 passed")

## Part 6 (coding task): Rebuild the readings

Going back is the same multiplication in reverse: $\hat{X} = Z W^\top + \boldsymbol{\mu}$, the scores times the transposed components, plus the mean. Write `reconstruct(Z, vecs, mu)`, which rebuilds the readings from scores on the first `Z.shape[1]` components, and `squared_error(X, X_hat)`, the total squared difference between the data and its reconstruction, as a float.

The lesson's identity is the check: the squared error of keeping $k$ components is $(n - 1)$ times the sum of the eigenvalues thrown away. For the four points and one component it is $3 \cdot 4/3 = 4$, and $(3, 1)$ and $(1, 3)$ both land on $(2, 2)$.

In [ ]:
def reconstruct(Z, vecs, mu):
    """The readings rebuilt from scores on the first Z.shape[1] components."""
    # TODO
    raise NotImplementedError


def squared_error(X, X_hat):
    """The total squared difference between X and its reconstruction."""
    # TODO
    raise NotImplementedError

In [ ]:
recon_small = reconstruct(z_small, vecs_small, mu_small)
print("the four points rebuilt from one component:\n", recon_small)
assert np.allclose(recon_small, [[0, 0], [2, 2], [2, 2], [2, 2]]), "from one component, (3, 1) and (1, 3) both land on (2, 2)"
assert np.isclose(squared_error(X_small, recon_small), 4), f"the total squared residual should be 4, got {squared_error(X_small, recon_small)}"

total = (n_hours - 1) * vals.sum()
errors = []
for k in range(1, 7):
    X_hat = reconstruct(project(Xc, vecs, k), vecs, mu)
    err = squared_error(X, X_hat)
    discarded = (n_hours - 1) * vals[k:].sum()
    errors.append(err)
    print(f"{k} components: squared error {err:10.3f}   (n - 1) x discarded eigenvalues {discarded:10.3f}")
    assert np.isclose(err, discarded), f"with {k} components the error should equal (n - 1) times the discarded eigenvalues"
assert errors[-1] < 1e-8, "all six components rebuild the data exactly"
assert errors[1] / total < 0.05, "two components should lose less than 5% of the variance"

X_hat2 = reconstruct(project(Xc, vecs, 2), vecs, mu)
plt.figure(figsize=(9, 3))
plt.plot(hour[:72], X[:72, 3], lw=1, label="sensor 3, measured")
plt.plot(hour[:72], X_hat2[:72, 3], lw=1.5, label="rebuilt from two components")
plt.xlabel("hour")
plt.ylabel("degrees C")
plt.legend()
plt.show()
print("Part 6 passed")

## Solution

### Part 1: Center the data

In [ ]:
def center(X):
    """(mu, Xc): the column means, and X with them subtracted from every row."""
    mu = X.mean(axis=0)
    return mu, X - mu

### Part 2: The covariance matrix

In [ ]:
def covariance(Xc):
    """The sample covariance matrix of centered data, divided by n - 1."""
    return Xc.T @ Xc / (len(Xc) - 1)

### Part 3: The principal components

In [ ]:
def principal_components(Sigma):
    """(vals, vecs): eigenvalues in descending order, unit eigenvectors as the columns of vecs."""
    vals, vecs = np.linalg.eigh(Sigma)
    order = np.argsort(vals)[::-1]
    return vals[order], vecs[:, order]

### Part 4: How much each component explains

In [ ]:
def explained(vals):
    """(fraction, cumulative): the share of variance per component and its running total."""
    fraction = vals / vals.sum()
    return fraction, np.cumsum(fraction)


def components_for(cumulative, target):
    """The smallest k whose first k components explain at least `target` of the variance."""
    return int(np.searchsorted(cumulative, target)) + 1

### Part 5: Two numbers an hour

In [ ]:
def project(Xc, vecs, k):
    """Scores of the centered rows on the first k components, an (n, k) matrix."""
    return Xc @ vecs[:, :k]

### Part 6: Rebuild the readings

In [ ]:
def reconstruct(Z, vecs, mu):
    """The readings rebuilt from scores on the first Z.shape[1] components."""
    k = Z.shape[1]
    return Z @ vecs[:, :k].T + mu


def squared_error(X, X_hat):
    """The total squared difference between X and its reconstruction."""
    return float(((X - X_hat) ** 2).sum())